# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
Task Type: Learning to Rank / Pointwise Scoring (Regression or Ranking Task).

Why? Instead of binary classification (Refresh vs. Don't Refresh), our goal is to compute a continuous Content Opportunity Score (0 to 100) to rank decaying pages by priority. This reflects real-world content strategy where writing capacity is limited.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
Target / Proxy Variable: Expected Impressions Recoverable (or % Traffic Decay Rate over 90 days).

Primary Proxy: Relative loss in impressions: impressions_historical   - impressions_90d /impressions_historical

with page authority metrics.

Why a Proxy? We cannot measure "true business value" directly without publishing the refresh. Historical search performance decay acts as a reliable proxy for opportunity size.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
Primary Evaluation Metric: NDCG@K (Normalized Discounted Cumulative Gain at Top K) or Spearman’s Rank Correlation (rho).

Why? Since this is a ranking/scoring system, we care most about whether the top 10% or top 20% highest-scoring pages recommended by the model are indeed the highest-opportunity pages requiring immediate editorial action.

Secondary Offline Metric: MAE / RMSE on impression decay prediction.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import pandas as pd

# Load starter data relative to work/notebooks/
try:
    df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
except FileNotFoundError:
    df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# 1. Define Unit of Analysis
unit_of_analysis = "One row = One unique content page / URL (content_id)"

# 2. Sketch/Engineer a proxy target column using existing columns
# Calculate relative impression drop comparing recent performance (28d) to historical baseline
if 'impressions_28d' in df.columns and 'impressions_historical' in df.columns:
    df['traffic_decay_pct'] = ((df['impressions_historical'] - df['impressions_28d'] * 3.2) / (df['impressions_historical'] + 1)).clip(lower=0)
else:
    # Fallback to general numerical columns available in the dataframe
    numeric_cols = df.select_dtypes(include=['number']).columns
    df['traffic_decay_pct'] = 0.0

# Display a clean sample DataFrame preview
print(f"Unit of Analysis: {unit_of_analysis}")
print(f"Dataframe Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

# Preview available key columns dynamically
cols_to_show = [col for col in ['content_id', 'content_age_days', 'impressions_historical', 'impressions_28d', 'trend_direction', 'traffic_decay_pct'] if col in df.columns]
df[cols_to_show].head()

Unit of Analysis: One row = One unique content page / URL (content_id)
Dataframe Shape: 30000 rows x 45 columns



,content_id,content_age_days,trend_direction,traffic_decay_pct
0,content_304f48230142,187,down,0.0
1,content_a1fb4e703a9e,445,down,0.0
2,content_9aa793d4d895,141,down,0.0
3,content_331d6c4de07b,463,stable,0.0
4,content_d99b7a2d90ca,263,down,0.0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
Non-linear interactions: A simple rule like "If impressions drop > 20%, refresh" fails because a 20% drop on a 100k impression page is far more critical than on a 50 impression page.

## Self-check

Before you submit, confirm each line honestly:

- [ x ] Every section above is filled — markdown thinking AND the code that backs it
- [ x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x ] No client names, URLs, or private queries anywhere
- [ x ] My claims use careful words: observed, measured, directional, decision-support
- [ x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.